# 02 - Condition だけを名寄せして curated h5ad + sidecar CSV を保存

このノートブックは **手入力の作業スペース** です。`pandas / numpy / scanpy / anndata / pathlib`
だけで動くシンプルな構成にしています。

## このノートブックの方針

- この 02 では **Condition の名寄せだけ** を行う。cell type / genotype / treatment /
  disease_status などの整理はまだ行わない。
- curated h5ad の `obs` は merge / QC に最低限必要な列だけにする。
- 元の `adata.obs` の全カラムは `original_obs_metadata_by_cell.csv`（sidecar CSV）に保存する。
- 細胞アノテーションなどは、後から `cell_uid` をキーに sidecar CSV から戻す。
- QC や可視化で必要になった列だけ sidecar CSV から追加する。
- `cell_uid` は `GSE番号 + "__" + 元の adata.obs_names` で作る。
- `cell_uid` は merge 後も sidecar CSV と結合するための **最重要キー** である。
- `Condition_original` は元データでの表記、`Condition` は解析用に揃えた表記である。

## curated h5ad の obs に残す列

```
cell_uid:
  全データセットを通して各細胞を一意に識別するID。
  元の adata.obs_names に GSE番号をprefixとして付けて作る。
  merge後や sidecar CSV との結合に使う最重要キー。
  例: GSE206330__AAACCTGAGATCGATA-1

source_accession:
  その細胞が由来するGSE番号。
  例: GSE206330, GSE173524

dataset_id:
  そのh5adファイルまたはデータセットを識別するID。
  基本的には入力h5adのファイル名stemを使う。
  例: GSE206330_sc_cortex_sod1_facs_glia_soupx

cell_id_original:
  元のh5adを読んだ時点の adata.obs_names。
  元データ内での細胞ID。
  GSE間では重複しうるので、mergeやsidecar結合には cell_uid を使う。

Condition_original:
  Condition名寄せに使った元の値。
  例: sample_label の "WT_1"、Condition列の "SOD1"、gsm_id の "GSM510xxxx" など。
  どの列を使うかは CURATION_RULES[acc]["condition_source"] で指定する。

Condition:
  解析で使うために名寄せした条件名。
  例: WT, Ctrl, SOD1, SMA, SOD1_RIPK1i, unknown。
  CURATION_RULES[acc]["condition_map"] で Condition_original から変換する。

sample_id:
  サンプル・個体・ライブラリなどを識別するID。
  既存obsに sample_id があれば使う。
  なければ "unknown"。

sample_label:
  元データや01で付いたサンプル名・ラベル。
  既存obsに sample_label があれば使う。
  なければ "unknown"。
```

以下の列は curated h5ad には追加せず、すべて sidecar CSV に逃がす:
`cell_type, cell_type_original, author_cell_type, author_condition, genotype,
treatment, disease_status, data_status`。

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad

# v2 ルートを探す（config/dataset_manifest.yaml がある場所）
ROOT = Path.cwd()
while not (ROOT / "config" / "dataset_manifest.yaml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

INTERIM_DIR = ROOT / "data" / "interim_h5ad"
CURATED_DIR = ROOT / "data" / "curated_h5ad"
CURATED_DIR.mkdir(parents=True, exist_ok=True)

print("project root:", ROOT)
print("interim dir :", INTERIM_DIR)
print("curated dir :", CURATED_DIR)

## interim h5ad の一覧

In [ ]:
interim_files = sorted(p for p in INTERIM_DIR.glob("*.h5ad"))
print(f"{len(interim_files)} interim h5ad files found:\n")
for p in interim_files:
    print(p.name)


def acc_of(path):
    """ファイル名の先頭 GSE 番号を accession として返す（例: GSE287569_xxx.h5ad -> GSE287569）。"""
    return path.stem.split("_")[0]


def load_interim_by_acc(acc):
    """accession（例: "GSE287569"）を prefix に持つ interim h5ad を読み込む。"""
    matches = sorted(p for p in INTERIM_DIR.glob(f"{acc}*.h5ad"))
    if not matches:
        raise FileNotFoundError(f"no interim h5ad starting with {acc!r} in {INTERIM_DIR}")
    if len(matches) > 1:
        print(f"[warn] multiple files match {acc!r}, using first: {[p.name for p in matches]}")
    return sc.read_h5ad(matches[0])

## obs 確認用の関数

In [ ]:
def inspect_obs(adata, max_values=50):
    print(adata)
    print("\nobs columns:")
    for i, col in enumerate(adata.obs.columns):
        print(f"{i}: {col}")

    for col in adata.obs.columns:
        s = adata.obs[col]
        n = s.dropna().astype(str).nunique()
        vals = s.dropna().astype(str).unique().tolist()[:max_values]
        print(f"\n--- {col} ---")
        print("n_unique:", n)
        print(vals)

## GSE を 1 つ指定して obs を確認する（作業セル）

`acc` を書き換えて、各 GSE の obs を順番に確認し、下の `CURATION_RULES` を埋めてください。

In [ ]:
acc = "GSE287569"
adata = load_interim_by_acc(acc)
inspect_obs(adata)

## CURATION_RULES（Condition 名寄せの手入力スペース）

`CURATION_RULES` は、各GSEでどのobs列をCondition名寄せに使うか、そして元の値をどの標準Condition名に変換するかを書く作業スペースである。

例:
```
  condition_source = "sample_label"
  condition_map = {"WT_1": "WT", "WT_2": "WT", "SOD1_1": "SOD1"}
```

この場合:
```
  Condition_original は WT_1, WT_2, SOD1_1
  Condition は WT, WT, SOD1
```

- `condition_source` が `None` のときは `Condition_original = "unknown"`, `Condition = "unknown"`。
- `condition_map` で map されなかった値は `"unknown"` にし、warning を表示する。
- 今回は完成させなくてよい（空辞書のままで構わない）。

In [ ]:
CURATION_RULES = {
    "GSE287569": {
        "condition_source": "sample_label",
        "condition_map": {
            # "s1": "WT",
            # "s2": "WT",
            # "s7": "SOD1_RIPK1i",
        },
    },

    "GSE173524": {
        "condition_source": "meta_Genotype",
        "condition_map": {
            # "WT": "WT",
            # "SOD1": "SOD1",
        },
    },

    "GSE167198": {
        "condition_source": None,
        "condition_map": {},
    },

    "GSE167327": {
        "condition_source": None,
        "condition_map": {},
    },

    "GSE167331": {
        "condition_source": None,
        "condition_map": {},
    },

    "GSE219201": {
        "condition_source": None,
        "condition_map": {},
    },

    "GSE242939": {
        "condition_source": None,
        "condition_map": {},
    },

    "GSE206330": {
        "condition_source": "Condition",
        "condition_map": {
            # "WT": "WT",
            # "SOD1": "SOD1",
        },
    },

    "GSE178693": {
        "condition_source": None,
        "condition_map": {},
    },

    "GSE295514": {
        "condition_source": None,
        "condition_map": {},
    },

    "GSE208629": {
        "condition_source": None,
        "condition_map": {},
    },
}

print(f"{len(CURATION_RULES)} GSE entries in CURATION_RULES")

## apply_curation（Condition のみ処理）

In [ ]:
def apply_curation(adata, acc, input_file, rules):
    adata = adata.copy()

    if not adata.obs_names.is_unique:
        raise ValueError(f"{acc}: adata.obs_names is not unique. Please fix this before curation.")

    dataset_id = input_file.stem
    cell_id_original = adata.obs_names.astype(str)
    cell_uid = acc + "__" + cell_id_original

    # Condition_original and Condition
    condition_source = rules.get(acc, {}).get("condition_source")
    condition_map = rules.get(acc, {}).get("condition_map", {})

    if condition_source is None:
        condition_original = pd.Series("unknown", index=adata.obs_names)
        condition_standardized = pd.Series("unknown", index=adata.obs_names)
    else:
        if condition_source not in adata.obs.columns:
            print(f"WARNING: {acc}: condition_source '{condition_source}' not found in obs")
            condition_original = pd.Series("unknown", index=adata.obs_names)
            condition_standardized = pd.Series("unknown", index=adata.obs_names)
        else:
            condition_original = adata.obs[condition_source].astype(str)
            condition_standardized = condition_original.map(condition_map).fillna("unknown")

            unmapped = sorted(condition_original[condition_standardized == "unknown"].dropna().unique().tolist())
            if unmapped:
                print(f"WARNING: {acc}: unmapped Condition_original values:", unmapped)

    sample_id = adata.obs["sample_id"].astype(str) if "sample_id" in adata.obs.columns else pd.Series("unknown", index=adata.obs_names)
    sample_label = adata.obs["sample_label"].astype(str) if "sample_label" in adata.obs.columns else pd.Series("unknown", index=adata.obs_names)

    minimal_obs = pd.DataFrame(index=adata.obs_names)
    minimal_obs["cell_uid"] = cell_uid
    minimal_obs["source_accession"] = acc
    minimal_obs["dataset_id"] = dataset_id
    minimal_obs["cell_id_original"] = cell_id_original
    minimal_obs["Condition_original"] = condition_original.values
    minimal_obs["Condition"] = condition_standardized.values
    minimal_obs["sample_id"] = sample_id.values
    minimal_obs["sample_label"] = sample_label.values

    adata.obs = minimal_obs
    adata.obs_names = adata.obs["cell_uid"].astype(str)

    return adata

## sidecar（元 obs 全カラム）を作る関数

In [ ]:
def build_sidecar(adata, acc, dataset_id):
    """元の adata.obs を全カラムそのまま残しつつ、結合キーを先頭に付けた sidecar を返す。

    adata は curation 前（元の obs_names を持つ状態）で渡すこと。
    """
    original_obs = adata.obs.copy()
    cell_id_original = adata.obs_names.astype(str)
    cell_uid = acc + "__" + cell_id_original

    sidecar = original_obs.copy()
    sidecar.insert(0, "cell_uid", cell_uid.values)
    sidecar.insert(1, "source_accession", acc)
    sidecar.insert(2, "dataset_id", dataset_id)
    sidecar.insert(3, "cell_id_original", cell_id_original.values)
    sidecar.insert(4, "obs_name_original", cell_id_original.values)
    sidecar.insert(5, "obs_name_curated", cell_uid.values)
    return sidecar

## 全 GSE に curation を適用して保存（curated h5ad + sidecar CSV）

In [ ]:
def _vals_str(values, limit=30):
    vals = sorted(pd.Series(values).astype(str).unique().tolist())
    shown = vals[:limit]
    s = ", ".join(shown)
    if len(vals) > limit:
        s += f", ... (+{len(vals) - limit} more)"
    return s


summary_rows = []
sidecar_rows = []

for p in interim_files:
    acc = acc_of(p)
    print(f"\n=== {acc}  ({p.name}) ===")

    adata = sc.read_h5ad(p)
    dataset_id = p.stem

    # sidecar は curation 前（元 obs）から作る
    sidecar_rows.append(build_sidecar(adata, acc, dataset_id))

    # Condition のみ名寄せ
    cur = apply_curation(adata, acc, p, CURATION_RULES)

    out_path = CURATED_DIR / p.name
    cur.write_h5ad(out_path)
    print(f"saved -> {out_path.name}  ({cur.n_obs} cells x {cur.n_vars} genes)")

    rule = CURATION_RULES.get(acc, {})
    obs = cur.obs
    summary_rows.append({
        "source_accession": acc,
        "input_file": p.name,
        "output_file": out_path.name,
        "n_cells": cur.n_obs,
        "n_genes": cur.n_vars,
        "condition_source": rule.get("condition_source"),
        "n_condition_original": obs["Condition_original"].nunique(),
        "condition_original_values": _vals_str(obs["Condition_original"]),
        "n_condition": obs["Condition"].nunique(),
        "condition_values": _vals_str(obs["Condition"]),
        "n_unknown_condition": int((obs["Condition"].astype(str) == "unknown").sum()),
    })

# sidecar CSV（元 obs 全カラム）を結合して保存
original_obs_metadata = pd.concat(sidecar_rows, axis=0, ignore_index=True)
sidecar_path = CURATED_DIR / "original_obs_metadata_by_cell.csv"
original_obs_metadata.to_csv(sidecar_path, index=False)
print(f"\nsidecar saved -> {sidecar_path.name}  "
      f"({original_obs_metadata.shape[0]} cells x {original_obs_metadata.shape[1]} cols)")

summary = pd.DataFrame(summary_rows)
print(f"done: {len(summary)} datasets curated")

## curation summary の表示と保存

In [ ]:
summary_path = CURATED_DIR / "curation_summary.csv"
summary.to_csv(summary_path, index=False)
print("saved:", summary_path)
summary

## sidecar CSV から元 obs 列を戻す関数

In [ ]:
def attach_original_obs(adata, sidecar_csv, key="cell_uid", columns=None, prefix="orig_"):
    """sidecar CSV から元 obs 列を、key（既定 cell_uid）で join して戻す。

    - columns=None なら sidecar の全列を戻す。columns=[...] なら指定列だけ。
    - 戻す列には prefix を付け、curated 側の列と衝突させない。
    - sidecar に無い列が指定されたら warning を出し、存在する列だけ戻す。
    - join できなかった細胞数を表示する。
    - 元の adata は破壊せず copy を返す。
    """
    adata = adata.copy()
    sidecar = pd.read_csv(sidecar_csv, dtype=str)

    if key not in sidecar.columns:
        raise ValueError(f"key '{key}' not found in sidecar CSV")
    if not sidecar[key].is_unique:
        raise ValueError(f"{key} is not unique in sidecar CSV")
    if key not in adata.obs.columns:
        raise ValueError(f"key '{key}' not found in adata.obs")

    # 戻す列を決める（key 自体は除く）
    available = [c for c in sidecar.columns if c != key]
    if columns is None:
        use_cols = available
    else:
        missing = [c for c in columns if c not in available]
        if missing:
            print("WARNING: columns not in sidecar CSV (skipped):", missing)
        use_cols = [c for c in columns if c in available]

    sidecar = sidecar.set_index(key)
    keys = adata.obs[key].astype(str)

    matched = keys.isin(sidecar.index)
    n_unmatched = int((~matched).sum())
    print(f"attach_original_obs: {int(matched.sum())} matched, {n_unmatched} unmatched of {adata.n_obs} cells")

    for c in use_cols:
        adata.obs[f"{prefix}{c}"] = sidecar[c].reindex(keys.values).values

    return adata

## 再付与テスト（sidecar から元 obs を戻せることの確認 / 保存はしない）

In [ ]:
sidecar_csv = CURATED_DIR / "original_obs_metadata_by_cell.csv"
test_file = sorted(CURATED_DIR.glob("*.h5ad"))[0]

test_adata = sc.read_h5ad(test_file)

# 全列を戻す
test_adata2 = attach_original_obs(
    test_adata,
    sidecar_csv=sidecar_csv,
    key="cell_uid",
    columns=None,
    prefix="orig_",
)

assert test_adata2.n_obs == test_adata.n_obs
assert "cell_uid" in test_adata2.obs.columns

orig_cols = [c for c in test_adata2.obs.columns if c.startswith("orig_")]
print("Number of restored original obs columns:", len(orig_cols))
print(orig_cols[:30])
display(test_adata2.obs[["cell_uid"] + orig_cols[:10]].head())

In [ ]:
# 細胞アノテーション候補だけ戻す例（存在しない列は warning のみ）
test_adata3 = attach_original_obs(
    test_adata,
    sidecar_csv=sidecar_csv,
    key="cell_uid",
    columns=[
        "cell_type",
        "meta_cell.type.refined",
        "meta_Genotype",
        "Condition",
        "sex",
        "subcluster",
        "nCount_RNA",
        "nFeature_RNA",
        "percent.mt",
    ],
    prefix="orig_",
)

display(test_adata3.obs.filter(like="orig_").head())

## (optional) merge preview

curated h5ad の `obs_names` はすでに `cell_uid` なので、追加の prefix は付けない（二重 prefix 防止）。**保存はしない**。

In [ ]:
# Optional: simple merge preview
curated_files = sorted(CURATED_DIR.glob("*.h5ad"))
adatas = [sc.read_h5ad(p) for p in curated_files]

merged = ad.concat(
    adatas,
    join="outer",
    label="source_file",
    keys=[p.stem for p in curated_files],
    index_unique=None,
)

print(merged)
merged.obs[["source_accession", "dataset_id", "Condition"]].value_counts().head(50)